# ML Development (COPD + ALT)

This notebook is used for **model development and comparison** before finalizing `src/ml/train.py`.

## Goals
- Analyze feature behavior with both targets.
- Add explicit outlier diagnostics and treatment strategy.
- Train and compare 2 candidate models per target.
- Save experimental metrics for decision-making.

## Targets
- `chronic_obstructive_pulmonary_disease` (multi-class: A/B/C/D)
- `alanine_aminotransferase` (regression)

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from scipy.stats import chi2_contingency, f_oneway, kruskal, zscore

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import settings
from data.loader import PatientDataLoader

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("CSV:", settings.patient_csv_path)
print("Artifacts:", settings.artifacts_dir)


In [ ]:
with PatientDataLoader(csv_path=settings.patient_csv_path) as loader:
    df = loader.get_dataframe().copy()

print(f"Shape: {df.shape}")
df.head()

In [ ]:
target_cls = "chronic_obstructive_pulmonary_disease"
target_reg = "alanine_aminotransferase"
id_col = "patient_id"
binary_categorical_cols = ["readmitted", "urban"]

feature_cols = [c for c in df.columns if c not in {target_cls, target_reg, id_col}]
all_numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

# Treat binary flags as categorical for EDA and preprocessing consistency.
num_cols = [c for c in all_numeric_cols if c not in binary_categorical_cols]
cat_cols = [c for c in feature_cols if c not in num_cols]

print("Features:", len(feature_cols))
print("Numeric (continuous) features:", num_cols)
print("Categorical features:", cat_cols)
#print("Binary categorical features:", binary_categorical_cols)

## 1) Univariate Analysis (Features)

Visual inspection of each feature before modeling.

In [ ]:
for col in [target_reg]+ num_cols:
    fig = px.histogram(df, x=col, nbins=30, title=f"Histogram — {col}")
    fig.show()

for col in [target_cls]+ cat_cols:
    counts = df[col].value_counts(dropna=False).reset_index()
    counts.columns = [col, "count"]
    fig = px.bar(counts, x=col, y="count", title=f"Count Plot — {col}")
    fig.show()

## 2) Outlier Diagnostics (Added)

This section identifies potential outliers for **continuous numeric features only** (`readmitted` and `urban` are treated as categorical).

Methods:
- IQR rule
- Absolute z-score > 3

Then we review boxplots and define treatment strategy.

In [ ]:
outlier_rows = []

for col in num_cols:
    series = df[col].astype(float)

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    iqr_mask = (series < lower) | (series > upper)

    z = np.abs(zscore(series, nan_policy="omit"))
    z_mask = z > 3

    outlier_rows.append(
        {
            "feature": col,
            "iqr_outliers": int(iqr_mask.sum()),
            "iqr_pct": float(iqr_mask.mean() * 100),
            "zscore_outliers": int(np.nansum(z_mask)),
            "zscore_pct": float(np.nanmean(z_mask) * 100),
            "lower_iqr_bound": float(lower),
            "upper_iqr_bound": float(upper),
        }
    )

outlier_df = pd.DataFrame(outlier_rows).sort_values("iqr_pct", ascending=False)
outlier_df

In [ ]:
for col in num_cols:
    fig = px.box(df, y=col, title=f"Boxplot — {col}")
    fig.show()

print("Outlier treatment recommendation:")
print("- Keep clinically plausible values (do not blindly drop).")
print("- If needed for linear models, winsorize only high-outlier columns.")
print("- Tree-based models are usually robust to outliers.")

IQR marks values outside Q1 − 1.5×IQR and Q3 + 1.5×IQR; z-score marks values more than 3 standard deviations from the mean (|z| > 3). In this dataset both are low: medication_count and days_hospitalized have the most (~1.2% IQR, ~0.4–0.5% z-score), then last_lab_glucose and bmi (~0.6–0.8% IQR), while age and albumin_globulin_ratio have none. So extreme values are rare; no aggressive outlier removal is needed for modeling, and tree-based models can keep the data as-is.

## 2.1) Optional Outlier Capping Check (Winsorization Preview)

Use this only for continuous features with high outlier ratio and only if it improves model stability.

In [ ]:
def winsorize_iqr(frame: pd.DataFrame, columns: list[str], factor: float = 1.5) -> pd.DataFrame:
    capped = frame.copy()
    for c in columns:
        s = capped[c].astype(float)
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr
        capped[c] = s.clip(lower=lower, upper=upper)
    return capped

high_outlier_features = outlier_df.loc[outlier_df["iqr_pct"] > 1.0, "feature"].tolist()
print("Candidate columns for optional capping:", high_outlier_features)

# Preview only, do not overwrite original df in development
# df_capped = winsorize_iqr(df, high_outlier_features)
# df_capped[high_outlier_features].describe().T.head()

## 3) Correlation and Association Analysis

- Numeric vs numeric: Pearson correlation.
- Categorical vs categorical: Cramer's V.
- Feature vs targets: statistical tests + visual checks.

In [ ]:
corr_num = df[num_cols + [target_reg]].corr(numeric_only=True)
fig = px.imshow(
    corr_num.to_numpy(dtype=float),
    x=list(corr_num.columns),
    y=list(corr_num.index),
    text_auto=True,
    aspect="auto",
    title="Continuous Numeric Correlation (Pearson)",
)
fig.show()


def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)
    return np.sqrt(phi2corr / max((kcorr - 1), (rcorr - 1), 1e-12))

cat_for_cramersv = cat_cols+[target_cls]
cat_assoc = pd.DataFrame(index=cat_for_cramersv, columns=cat_for_cramersv, dtype=float)
for c1 in cat_for_cramersv:
    for c2 in cat_for_cramersv:
        cat_assoc.loc[c1, c2] = cramers_v(df[c1], df[c2])

# Pass numpy + explicit labels: px.imshow errors when DataFrame index
# labels match column names (DuplicateError for readmitted/urban).
fig = px.imshow(
    cat_assoc.to_numpy(dtype=float),
    x=list(cat_assoc.columns),
    y=list(cat_assoc.index),
    text_auto=True,
    aspect="auto",
    title="Categorical Association (Cramer's V)",
)
fig.show()


In [ ]:
# Categorical features vs COPD (chi-square)
chi_results = []
for col in cat_cols:
    table = pd.crosstab(df[col], df[target_cls])
    chi2, p, _, _ = chi2_contingency(table)
    chi_results.append({"feature": col, "chi2": chi2, "p_value": p})

chi_df = pd.DataFrame(chi_results).sort_values("p_value")
chi_df



In [ ]:
# Continuous numeric features vs COPD (Kruskal-Wallis)
kw_results = []
for col in num_cols:
    groups = [g[col].values for _, g in df.groupby(target_cls)]
    stat, p = kruskal(*groups)
    kw_results.append({"feature": col, "kw_stat": stat, "p_value": p})

pd.DataFrame(kw_results).sort_values("p_value")

In [ ]:
# ALT associations
anova_rows = []
for col in cat_cols:
    groups = [g[target_reg].values for _, g in df.groupby(col)]
    stat, p = f_oneway(*groups)
    anova_rows.append({"feature": col, "f_stat": stat, "p_value": p})

anova_df = pd.DataFrame(anova_rows).sort_values("p_value")
anova_df

for col in ["sex", "smoker", "exercise_frequency", "diet_quality"]:
    fig = px.box(df, x=col, y=target_reg, title=f"{target_reg} by {col}")
    fig.show()

## 4) Feature Filtering and Selection Heuristics

Rules used in development phase:
- Exclude `patient_id`.
- Remove near-constant columns if any.
- Inspect very high numeric collinearity (>0.95).
- Rank features by mutual information with each target.

In [ ]:
X = df[feature_cols].copy()
y_cls = df[target_cls].copy()
y_reg = df[target_reg].copy()

# quick multicollinearity check for continuous numeric features
high_corr_pairs = []
num_corr = X[num_cols].corr().abs()
for i, c1 in enumerate(num_cols):
    for c2 in num_cols[i + 1:]:
        v = num_corr.loc[c1, c2]
        if v > 0.95:
            high_corr_pairs.append((c1, c2, float(v)))

high_corr_pairs_df = pd.DataFrame(high_corr_pairs, columns=["feature_1", "feature_2", "abs_corr"]) if high_corr_pairs else pd.DataFrame(columns=["feature_1", "feature_2", "abs_corr"])
high_corr_pairs_df

# MI ranking for COPD (requires encoded categoricals)
X_encoded_cls = pd.get_dummies(X, columns=cat_cols, drop_first=False)
mi_cls = mutual_info_classif(X_encoded_cls, y_cls, random_state=RANDOM_STATE)
mi_cls_df = pd.DataFrame({"feature": X_encoded_cls.columns, "mi_cls": mi_cls}).sort_values("mi_cls", ascending=False)

# MI ranking for ALT
mi_reg = mutual_info_regression(X_encoded_cls, y_reg, random_state=RANDOM_STATE)
mi_reg_df = pd.DataFrame({"feature": X_encoded_cls.columns, "mi_reg": mi_reg}).sort_values("mi_reg", ascending=False)

display(mi_cls_df.head(20))
display(mi_reg_df.head(20))

## 5) Train Candidate Models (2 per Target)

- COPD: `LogisticRegression` vs `RandomForestClassifier`
- ALT: `Ridge` vs `RandomForestRegressor`

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# Classification: stratified split (balanced COPD classes in train/test)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X,
    y_cls,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_cls,
)

# Regression: random split without stratification
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

cls_models = {
    "logreg": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "rf_cls": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
    ]),
}

reg_models = {
    "ridge": Pipeline([
        ("prep", preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]),
    "rf_reg": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)),
    ]),
}

cls_results = []
for name, model in cls_models.items():
    model.fit(X_train_cls, y_train_cls)
    pred = model.predict(X_test_cls)
    cls_results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test_cls, pred),
            "macro_f1": f1_score(y_test_cls, pred, average="macro"),
        }
    )

reg_results = []
for name, model in reg_models.items():
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)
    reg_results.append(
        {
            "model": name,
            "mae": mean_absolute_error(y_test_reg, pred),
            "rmse": root_mean_squared_error(y_test_reg, pred),
            "r2": r2_score(y_test_reg, pred),
        }
    )

cls_results_df = pd.DataFrame(cls_results).sort_values("macro_f1", ascending=False)
reg_results_df = pd.DataFrame(reg_results).sort_values("rmse")

display(cls_results_df)
display(reg_results_df)



In [ ]:
best_cls_name = cls_results_df.iloc[0]["model"]
best_reg_name = reg_results_df.iloc[0]["model"]

best_cls = cls_models[best_cls_name]
best_reg = reg_models[best_reg_name]

best_cls.fit(X_train_cls, y_train_cls)
best_reg.fit(X_train_reg, y_train_reg)

pred_cls = best_cls.predict(X_test_cls)
pred_reg = best_reg.predict(X_test_reg)

print(f"Best COPD model: {best_cls_name}")
print(classification_report(y_test_cls, pred_cls))

cm = confusion_matrix(y_test_cls, pred_cls, labels=sorted(y_cls.unique()))
fig = px.imshow(
    cm,
    x=sorted(y_cls.unique()),
    y=sorted(y_cls.unique()),
    text_auto=True,
    title=f"Confusion Matrix — {best_cls_name}",
)
fig.show()

print(f"Best ALT model: {best_reg_name}")
print("MAE:", mean_absolute_error(y_test_reg, pred_reg))
print("RMSE:", root_mean_squared_error(y_test_reg, pred_reg))
print("R2:", r2_score(y_test_reg, pred_reg))



In [ ]:
# Optional: save development metrics as artifact for traceability
#insights_dir = settings.artifacts_dir / "insights"
#insights_dir.mkdir(parents=True, exist_ok=True)

#summary = {
#    "best_models": {
#        "copd": best_cls_name,
#        "alt": best_reg_name,
#    },
#    "classification_candidates": cls_results,
#    "regression_candidates": reg_results,
#    "outlier_summary_top": outlier_df.head(10).to_dict(orient="records"),
#}

#output_path = insights_dir / "ml_development_summary.json"
#with output_path.open("w", encoding="utf-8") as f:
#    json.dump(summary, f, indent=2)

#print("Saved:", output_path)

## 6) Conclution

### 1. Selected model per target and why

| Target | Candidate results | Decision | Rationale |
|--------|-------------------|----------|-----------|
| **COPD** (4-class) | LogReg: acc 0.248, macro F1 0.247; RF: acc 0.252, macro F1 0.251 | **Try LightGBM/XGBoost next** before finalizing; interim pick = `rf_cls` (marginally better) | Both baselines are at **random-chance level** (~25% for 4 balanced classes). Neither LogReg nor RF beats the baseline meaningfully. |
| **ALT** (regression) | Ridge: MAE 0.082, RMSE 0.102, R² 0.9996; RF: MAE 0.088, RMSE 0.113, R² 0.9995 | **`ridge`** | Slightly better metrics, simpler and more interpretable. Full model performs almost identically to **BMI-only** prediction (MAE 0.081 vs 0.082). |

**Train/test splits:**
- COPD: stratified split (`stratify=y_cls`) — balanced A/B/C/D in train and test
- ALT: random split (no stratification)



### 2. Features excluded and why

| Feature | Status | Rationale |
|---------|--------|-----------|
| `patient_id` | **Excluded** | Identifier only; no predictive value |
| All other 15 features | **Kept** | No strong multicollinearity (no pairs with abs(r) > 0.95). Univariate tests were weak for most features, but dropping before modeling is premature — multivariate models may still use combinations |

**Preprocessing:** 6 continuous features → `StandardScaler`; 9 categorical (incl. `readmitted`, `urban`) → `OneHotEncoder` → 33 model inputs total.



### 3. Outlier treatment decision

**Decision: no removal, no winsorization for POC.**

- IQR and z-score outlier rates are low (< ~1.3% on all continuous features)
- `age` and `albumin_globulin_ratio`: zero outliers by both methods
- Highest: `medication_count`, `days_hospitalized` (~1.2% IQR)
- Tree models are robust; linear models (Ridge) did not show outlier-related issues on ALT



### 4. Key findings from EDA (context for model results)

| Target | Strongest univariate signals | Model implication |
|--------|------------------------------|-------------------|
| COPD | `diet_quality` (χ² p ≈ 0.036); MI tops: `income_bracket`, `urban` | Weak overall signal → explains ~25% accuracy |
| ALT | `bmi` (Pearson ≈ 0.9998 with ALT); `readmitted` (ANOVA p ≈ 0.01) | ALT is practically **BMI-driven** in this dataset |



### 5. Model quality assessment

**COPD — not usable yet at baseline level**
- Accuracy and macro F1 ≈ 0.25 = random guessing for 4 classes
- LogReg and RF nearly identical → features are not providing clear class separation
- Aligns with EDA: only `diet_quality` significant in χ²; Kruskal–Wallis non-significant for all continuous features

**ALT — excellent metrics, but trivial**
- R² ≈ 0.9996, MAE ≈ 0.08 on ALT range ~10–44
- **BMI alone** achieves the same performance as the full Ridge model
- Other features add negligible value; this is a characteristic of the synthetic dataset, not proof of a complex ML success

ALT prediction is effectively BMI-driven; COPD remains a hard target where baseline models do not beat random chance 

## 7) Train Phase 2 — Extended COPD Candidates (4 models)

Same splits as section 5. **COPD:** keep Phase 1 models and add boosting with class balancing.

| Target | Models |
|--------|--------|
| **COPD** | `logreg`, `rf_cls` (as in Phase 1) + `lgbm_cls`, `xgb_cls` (**balanced class weights**) |
| **ALT** | `ridge`, `rf_reg`, `xgb_reg` (gradient boosting regressor) |

**macOS:** if LightGBM/XGBoost import fails, run `brew install libomp` and restart kernel.


In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier, XGBRegressor
from sklearn.utils.class_weight import compute_sample_weight

# XGBoost requires numeric class labels (0..n-1), not COPD strings A/B/C/D
copd_label_encoder_p2 = LabelEncoder()
y_train_cls_xgb_p2 = copd_label_encoder_p2.fit_transform(y_train_cls)

preprocessor_p2 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

cls_models_p2 = {
    "logreg": Pipeline([
        ("prep", preprocessor_p2),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "rf_cls": Pipeline([
        ("prep", preprocessor_p2),
        ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
    ]),
    "lgbm_cls": Pipeline([
        ("prep", preprocessor_p2),
        (
            "model",
            LGBMClassifier(
                n_estimators=300,
                learning_rate=0.05,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
            ),
        ),
    ]),
    "xgb_cls": Pipeline([
        ("prep", preprocessor_p2),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                objective="multi:softmax",
                num_class=len(copd_label_encoder_p2.classes_),
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
                verbosity=0,
            ),
        ),
    ]),
}

reg_models_p2 = {
    "ridge": Pipeline([
        ("prep", preprocessor_p2),
        ("model", Ridge(alpha=1.0)),
    ]),
    "rf_reg": Pipeline([
        ("prep", preprocessor_p2),
        ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)),
    ]),
    "xgb_reg": Pipeline([
        ("prep", preprocessor_p2),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                learning_rate=0.05,
                random_state=RANDOM_STATE,
                verbosity=0,
            ),
        ),
    ]),
}

cls_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train_cls)

cls_results_p2 = []
for name, model in cls_models_p2.items():
    if name == "xgb_cls":
        model.fit(X_train_cls, y_train_cls_xgb_p2, model__sample_weight=cls_sample_weight)
        pred = copd_label_encoder_p2.inverse_transform(model.predict(X_test_cls).astype(int))
    else:
        model.fit(X_train_cls, y_train_cls)
        pred = model.predict(X_test_cls)

    cls_results_p2.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test_cls, pred),
            "macro_f1": f1_score(y_test_cls, pred, average="macro"),
        }
    )

reg_results_p2 = []
for name, model in reg_models_p2.items():
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)
    reg_results_p2.append(
        {
            "model": name,
            "mae": mean_absolute_error(y_test_reg, pred),
            "rmse": root_mean_squared_error(y_test_reg, pred),
            "r2": r2_score(y_test_reg, pred),
        }
    )

cls_results_p2_df = pd.DataFrame(cls_results_p2).sort_values("macro_f1", ascending=False)
reg_results_p2_df = pd.DataFrame(reg_results_p2).sort_values("rmse")

print("COPD candidates (Phase 2) — random baseline: accuracy=0.25, macro_f1~=0.25")
display(cls_results_p2_df)
display(reg_results_p2_df)


In [ ]:
best_cls_name_p2 = cls_results_p2_df.iloc[0]["model"]
best_reg_name_p2 = reg_results_p2_df.iloc[0]["model"]
best_cls_p2 = cls_models_p2[best_cls_name_p2]

if best_cls_name_p2 == "xgb_cls":
    best_cls_p2.fit(X_train_cls, y_train_cls_xgb_p2, model__sample_weight=cls_sample_weight)
    pred_cls_p2 = copd_label_encoder_p2.inverse_transform(best_cls_p2.predict(X_test_cls).astype(int))
else:
    best_cls_p2.fit(X_train_cls, y_train_cls)
    pred_cls_p2 = best_cls_p2.predict(X_test_cls)

best_cls_acc_p2 = accuracy_score(y_test_cls, pred_cls_p2)
best_cls_f1_p2 = f1_score(y_test_cls, pred_cls_p2, average="macro")

print(f"Best COPD Phase 2 model: {best_cls_name_p2}")
print(classification_report(y_test_cls, pred_cls_p2))

RANDOM_BASELINE_ACC = 0.25
COPD_LIMIT_MARGIN = 0.03

if best_cls_acc_p2 < RANDOM_BASELINE_ACC + COPD_LIMIT_MARGIN:
    print(
        "\nCOPD LIMITATION: even with 4 candidates (incl. balanced LightGBM/XGBoost), "
        f"best accuracy={best_cls_acc_p2:.3f} remains near random baseline (~0.25). "
        "Document this limitation in train.py / API disclaimer."
    )
else:
    print(
        f"\nCOPD Phase 2 improved above baseline margin: "
        f"accuracy={best_cls_acc_p2:.3f}, macro_f1={best_cls_f1_p2:.3f}"
    )


In [ ]:
#insights_dir = settings.artifacts_dir / "insights"
#insights_dir.mkdir(parents=True, exist_ok=True)

#near_random_p2 = best_cls_acc_p2 < RANDOM_BASELINE_ACC + COPD_LIMIT_MARGIN
#phase2_summary = {
#    "phase": 2,
#    "best_models": {"copd": best_cls_name_p2, "alt": best_reg_name_p2},
#    "classification_candidates": cls_results_p2,
#    "regression_candidates": reg_results_p2,
#    "copd_near_random_baseline": near_random_p2,
#}
#with (insights_dir / "ml_development_phase2_summary.json").open("w", encoding="utf-8") as f:
#    json.dump(phase2_summary, f, indent=2)
#print("Saved:", insights_dir / "ml_development_phase2_summary.json")


## 8) Train Phase 3 — Target-Specific Reduced Feature Sets

Phase 2 model lineup, but **separate 6-feature sets** per target (from EDA).

### COPD classification features (6)

| Feature | Type | Why |
|---------|------|-----|
| `diet_quality` | categorical | Only feature with significant χ² vs COPD (p≈0.036) |
| `income_bracket` | categorical | Highest mutual information for COPD |
| `urban` | categorical | Strong COPD MI; socioeconomic / care-access proxy |
| `diagnosis_code` | categorical | Clinical grouping + solid COPD MI |
| `exercise_frequency` | categorical | Lifestyle signal in COPD MI ranking |
| `smoker` | categorical | Clinically relevant for pulmonary outcomes |

### ALT regression features (6)

| Feature | Type | Why |
|---------|------|-----|
| `bmi` | numeric | Dominant ALT predictor (Pearson ~0.9998) |
| `readmitted` | categorical | Only categorical with significant ANOVA vs ALT (p≈0.01) |
| `exercise_frequency` | categorical | Top non-BMI MI for ALT |
| `albumin_globulin_ratio` | numeric | Lab marker; 2nd-tier MI for ALT |
| `diagnosis_code` | categorical | Clinical context for liver/metabolic patterns |
| `diet_quality` | categorical | Lifestyle factor with moderate ALT MI |

**Models:** COPD → `logreg`, `rf_cls`, `lgbm_cls` (balanced), `xgb_cls` (balanced + label encoding) | ALT → `ridge`, `rf_reg`, `xgb_reg`


In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier, XGBRegressor
from sklearn.utils.class_weight import compute_sample_weight

# --- COPD feature subset (classification) ---
phase3_cls_feature_cols = [
    "diet_quality",
    "income_bracket",
    "urban",
    "diagnosis_code",
    "exercise_frequency",
    "smoker",
]
phase3_cls_num_cols: list[str] = []
phase3_cls_cat_cols = phase3_cls_feature_cols.copy()

X_cls_p3 = df[phase3_cls_feature_cols].copy()

X_train_cls_p3, X_test_cls_p3, y_train_cls_p3, y_test_cls_p3 = train_test_split(
    X_cls_p3,
    y_cls,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_cls,
)

# XGBoost needs numeric labels — define encoder before model dict
copd_label_encoder_p3 = LabelEncoder()
y_train_cls_xgb_p3 = copd_label_encoder_p3.fit_transform(y_train_cls_p3)
cls_sample_weight_p3 = compute_sample_weight(
    class_weight="balanced", y=y_train_cls_p3
)

preprocessor_cls_p3 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), phase3_cls_cat_cols),
    ]
)

cls_models_p3 = {
    "logreg": Pipeline([
        ("prep", preprocessor_cls_p3),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "rf_cls": Pipeline([
        ("prep", preprocessor_cls_p3),
        ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
    ]),
    "lgbm_cls": Pipeline([
        ("prep", preprocessor_cls_p3),
        (
            "model",
            LGBMClassifier(
                n_estimators=300,
                learning_rate=0.05,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
            ),
        ),
    ]),
    "xgb_cls": Pipeline([
        ("prep", preprocessor_cls_p3),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                objective="multi:softmax",
                num_class=len(copd_label_encoder_p3.classes_),
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
                verbosity=0,
            ),
        ),
    ]),
}

cls_results_p3 = []
for name, model in cls_models_p3.items():
    if name == "xgb_cls":
        model.fit(
            X_train_cls_p3,
            y_train_cls_xgb_p3,
            model__sample_weight=cls_sample_weight_p3,
        )
        pred = copd_label_encoder_p3.inverse_transform(
            model.predict(X_test_cls_p3).astype(int)
        )
    else:
        model.fit(X_train_cls_p3, y_train_cls_p3)
        pred = model.predict(X_test_cls_p3)

    cls_results_p3.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test_cls_p3, pred),
            "macro_f1": f1_score(y_test_cls_p3, pred, average="macro"),
        }
    )

cls_results_p3_df = pd.DataFrame(cls_results_p3).sort_values("macro_f1", ascending=False)

# --- ALT feature subset (regression) ---
phase3_reg_feature_cols = [
    "bmi",
    "readmitted",
    "exercise_frequency",
    "albumin_globulin_ratio",
    "diagnosis_code",
    "diet_quality",
]
phase3_reg_num_cols = ["bmi", "albumin_globulin_ratio"]
phase3_reg_cat_cols = [
    "readmitted",
    "exercise_frequency",
    "diagnosis_code",
    "diet_quality",
]

X_reg_p3 = df[phase3_reg_feature_cols].copy()

X_train_reg_p3, X_test_reg_p3, y_train_reg_p3, y_test_reg_p3 = train_test_split(
    X_reg_p3,
    y_reg,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

preprocessor_reg_p3 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), phase3_reg_num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), phase3_reg_cat_cols),
    ]
)

reg_models_p3 = {
    "ridge": Pipeline([
        ("prep", preprocessor_reg_p3),
        ("model", Ridge(alpha=1.0)),
    ]),
    "rf_reg": Pipeline([
        ("prep", preprocessor_reg_p3),
        ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)),
    ]),
    "xgb_reg": Pipeline([
        ("prep", preprocessor_reg_p3),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                learning_rate=0.05,
                random_state=RANDOM_STATE,
                verbosity=0,
            ),
        ),
    ]),
}

reg_results_p3 = []
for name, model in reg_models_p3.items():
    model.fit(X_train_reg_p3, y_train_reg_p3)
    pred = model.predict(X_test_reg_p3)
    reg_results_p3.append(
        {
            "model": name,
            "mae": mean_absolute_error(y_test_reg_p3, pred),
            "rmse": root_mean_squared_error(y_test_reg_p3, pred),
            "r2": r2_score(y_test_reg_p3, pred),
        }
    )

reg_results_p3_df = pd.DataFrame(reg_results_p3).sort_values("rmse")

print("Phase 3 COPD features:", phase3_cls_feature_cols)
print("Phase 3 ALT features:", phase3_reg_feature_cols)
print("COPD candidates (Phase 3) — random baseline: accuracy=0.25")
display(cls_results_p3_df)
display(reg_results_p3_df)

print("\n--- Phase 2 vs Phase 3 (best COPD macro F1) ---")
try:
    print(
        f"Phase 2 best: {best_cls_name_p2}, "
        f"macro_f1={cls_results_p2_df.iloc[0]['macro_f1']:.4f}"
    )
except NameError:
    print("Run Phase 2 first for comparison.")
print(
    f"Phase 3 best: {cls_results_p3_df.iloc[0]['model']}, "
    f"macro_f1={cls_results_p3_df.iloc[0]['macro_f1']:.4f}"
)


In [ ]:
RANDOM_BASELINE_ACC = 0.25
COPD_LIMIT_MARGIN = 0.03

best_cls_name_p3 = cls_results_p3_df.iloc[0]["model"]
best_reg_name_p3 = reg_results_p3_df.iloc[0]["model"]

print(f"Selected Phase 3 COPD model: {best_cls_name_p3}")
print(f"Selected Phase 3 ALT model: {best_reg_name_p3}")

near_random_p3 = cls_results_p3_df.iloc[0]["accuracy"] < RANDOM_BASELINE_ACC + COPD_LIMIT_MARGIN
if near_random_p3:
    print(
        "COPD LIMITATION (Phase 3): target-specific 6-feature set still near random baseline. "
        "Consider full feature set for final COPD model."
    )

phase3_summary = {
    "phase": 3,
    "copd_features": phase3_cls_feature_cols,
    "copd_feature_rationale": {
        "diet_quality": "Only significant chi-square vs COPD",
        "income_bracket": "Highest MI for COPD",
        "urban": "Strong COPD MI; socioeconomic proxy",
        "diagnosis_code": "Clinical grouping + COPD MI",
        "exercise_frequency": "Lifestyle signal in COPD MI",
        "smoker": "Clinically relevant for pulmonary outcomes",
    },
    "alt_features": phase3_reg_feature_cols,
    "alt_feature_rationale": {
        "bmi": "Dominant ALT predictor (Pearson ~0.9998)",
        "readmitted": "Significant ANOVA vs ALT",
        "exercise_frequency": "Top non-BMI MI for ALT",
        "albumin_globulin_ratio": "Lab marker; 2nd-tier ALT MI",
        "diagnosis_code": "Clinical context",
        "diet_quality": "Lifestyle factor with moderate ALT MI",
    },
    "best_models": {"copd": best_cls_name_p3, "alt": best_reg_name_p3},
    "classification_candidates": cls_results_p3,
    "regression_candidates": reg_results_p3,
    "copd_near_random_baseline": near_random_p3,
}

#insights_dir = settings.artifacts_dir / "insights"
#with (insights_dir / "ml_development_phase3_summary.json").open("w", encoding="utf-8") as f:
#    json.dump(phase3_summary, f, indent=2)
#print("Saved:", insights_dir / "ml_development_phase3_summary.json")


## 9) Final Decision

### Selected final models

| Target | Model | Features |
|--------|-------|----------|
| **COPD** (4-class) | `XGBClassifier` (balanced sample weights + `LabelEncoder` for A/B/C/D) | 6 classification features from Phase 3 |
| **ALT** (regression) | `Ridge` | 6 regression features from Phase 3 |

**Why not `rf_reg` for ALT?** Ridge has better metrics (lower MAE/RMSE), is simpler, and easier to explain in the interview.

**Why 6 features per target (not 15)?** Phase 3 target-specific subsets are justified by EDA and keep the POC focused. For ALT, performance is still effectively **BMI-driven** (6-feature Ridge ≈ BMI-only Ridge).

### COPD features (classification)

`diet_quality`, `income_bracket`, `urban`, `diagnosis_code`, `exercise_frequency`, `smoker`

### ALT features (regression)

`bmi`, `readmitted`, `exercise_frequency`, `albumin_globulin_ratio`, `diagnosis_code`, `diet_quality`



### Known limitations 

1. **COPD:** accuracy and macro F1 remain near **random baseline (~0.25)** even with XGBoost and class balancing. This is a **data signal limitation**, not a tuning gap. Add disclaimer in Prediction Agent responses.
2. **ALT:** excellent R² (~0.9996) but largely explained by **BMI** — other features add marginal value.